<a href="https://colab.research.google.com/github/GTimothee/transformers/blob/sentencetransformers_load_peft/bug1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets
!pip install --upgrade sentence-transformers
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

# case where we want to add tokens

In [ ]:
# from sentence_transformers import SentenceTransformer
# from peft import LoraConfig, TaskType

# test_new_tokens = ["[bla]", "[blub]", "[bloo]", "[blib]"]

# # Create a resized base model
# model = SentenceTransformer("all-MiniLM-L6-v2")
# model.tokenizer.add_tokens(test_new_tokens)
# model[0].auto_model.resize_token_embeddings(len(model.tokenizer))
# model.save_pretrained("all-MiniLM-L6-v2-resized")

# Training

code mostly copied from https://www.philschmid.de/fine-tune-embedding-model-for-rag

In [2]:
from datasets import load_dataset

# Load dataset from the hub
dataset = load_dataset("philschmid/finanical-rag-embedding-dataset", split="train")

# rename columns
dataset = dataset.rename_column("question", "anchor")
dataset = dataset.rename_column("context", "positive")

# Add an id column to the dataset
dataset = dataset.add_column("id", range(len(dataset)))

# split dataset into a 10% test set
dataset = dataset.train_test_split(test_size=0.1)

# save datasets to disk
dataset["train"].to_json("train_dataset.json", orient="records")
dataset["test"].to_json("test_dataset.json", orient="records")

README.md:   0%|          | 0.00/882 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

247654

In [3]:
from datasets import load_dataset, concatenate_datasets
train_dataset = load_dataset("json", data_files="train_dataset.json")['train'].select(range(500))
val_dataset = load_dataset("json", data_files="test_dataset.json")['train']
corpus_dataset = concatenate_datasets([train_dataset, val_dataset])

corpus = dict(
    zip(corpus_dataset["id"], corpus_dataset["positive"])
)  # Our corpus (cid => document)
queries = dict(
    zip(val_dataset["id"], val_dataset["anchor"])
)  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for q_id in queries:
    relevant_docs[q_id] = [q_id]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator
)
from sentence_transformers.util import cos_sim
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    score_functions={"cosine": cos_sim},
    batch_size=32,
    corpus_chunk_size=100,
    show_progress_bar=True
)

In [5]:
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType
import torch

# Load the resized model and add an adapter
# model = SentenceTransformer("all-MiniLM-L6-v2-resized")
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)
model.add_adapter(peft_config)

embedding = model.encode("[bla] my name is [blub]")
print(embedding[:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-0.05346014 -0.02178512 -0.00050324 -0.02896873 -0.09822001  0.00544652
  0.19883959  0.00645556 -0.04425739 -0.09591077]


In [6]:
#results = ir_evaluator(model)
#results

In [7]:
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

args = SentenceTransformerTrainingArguments(
    output_dir="training_trial",
    num_train_epochs=.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=.5,
    warmup_ratio=0.1,

    learning_rate=1e-5,
    weight_decay=0.001,
    fp16=True,
    lr_scheduler_type='reduce_lr_on_plateau',
    report_to=None,

    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=.2,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_cosine_ndcg@10",
)

In [8]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset.select_columns(
        ["anchor", "positive"]
    ),
    loss=MultipleNegativesRankingLoss(model),
    evaluator=ir_evaluator
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [9]:
from transformers.utils.import_utils import is_peft_available
is_peft_available()

True

In [10]:
from packaging import version
import importlib
version.parse(importlib.metadata.version("peft"))

<Version('0.14.0')>

In [11]:
from transformers.integrations.peft import PeftAdapterMixin
print(isinstance(trainer.model, PeftAdapterMixin))
isinstance(trainer.model[0].auto_model, PeftAdapterMixin)

False


True

In [12]:
import os
os.environ["WANDB_MODE"] = "disabled"
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
0,0.004900,No log,0.774286,0.878571,0.911429,0.932857,0.774286,0.292857,0.182286,0.093286,0.774286,0.878571,0.911429,0.932857,0.854553,0.829133,0.832121


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:  92%|█████████▏| 11/12 [00:01<00:00,  8.11it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 12/12 [00:01<00:00,  7.66it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'training_trial/checkpoint-25/pytorch_model.bin'

In [ ]:
trainer.model[0].auto_model.active_adapters()

['default']

In [ ]:
# # Save the adapter itself
# model.save_pretrained("all-MiniLM-L6-v2-adapter")
# # Load the adapter directly
# loaded_model = SentenceTransformer("all-MiniLM-L6-v2-adapter")
# embedding = loaded_model.encode("[bla] my name is [blub]")
# print(embedding[:10])